In [7]:
from datetime import datetime, timedelta

# =========================================================
# [1] 함수 정의 (백엔드 로직)
# =========================================================
def get_expiry_recommendations(user_pantry_list, all_recipes_list):
    """
    유통기한 임박 재료 가중치 추천 알고리즘
    """
    today = datetime.now().date()
    pantry_scores = {}

    # 1. 내 냉장고 재료 점수화 (유통기한이 짧을수록 고득점)
    for item in user_pantry_list:
        # 날짜 처리
        exp_date = item['expires_at']
        if isinstance(exp_date, str):
            exp_date = datetime.strptime(exp_date, '%Y-%m-%d').date()
            
        days_left = (exp_date - today).days
        if days_left < 0: days_left = 0 # 만료됨 (0일로 처리)
        
        # ⭐️ 핵심 공식: 10점 / (남은일수 + 1)
        urgency_score = 10 / (days_left + 1)
        pantry_scores[item['ingredient_name']] = urgency_score

    # 2. 레시피별 점수 합산
    scored_recipes = []
    
    for recipe in all_recipes_list:
        total_score = 0
        matched_details = []
        
        # 레시피 재료 순회
        for ing_name in recipe['ingredients']:
            # 내 냉장고 재료와 매칭 (부분 일치 허용)
            for my_ing, score in pantry_scores.items():
                if my_ing in ing_name or ing_name in my_ing:
                    total_score += score
                    matched_details.append(f"{ing_name}({score:.1f}점)")
                    break 
        
        if total_score > 0:
            scored_recipes.append({
                'title': recipe['title'],
                'total_score': round(total_score, 2),
                'matched_reason': ", ".join(matched_details)
            })

    # 3. 점수 높은 순 정렬
    scored_recipes.sort(key=lambda x: x['total_score'], reverse=True)
    return scored_recipes


# =========================================================
# [2] 실행 및 호출 코드 (테스트용 데이터 포함)
# =========================================================
if __name__ == "__main__":
    # 1. 가상의 DB 데이터 준비
    today = datetime.now().date()
    
    # 내 냉장고 (콩나물/어묵: 오늘 만료, 닭가슴살: 3일 남음)
    my_pantry_db = [
        {'ingredient_name': '콩나물', 'expires_at': str(today)},           
        {'ingredient_name': '판어묵', 'expires_at': str(today)},             
        {'ingredient_name': '닭가슴살', 'expires_at': str(today + timedelta(days=3))}, 
    ]

    # 전체 레시피 DB
    all_recipes_db = [
        {'id': 1, 'title': '매운 어묵 콩나물찜', 'ingredients': ['어묵', '콩나물', '대파', '고춧가루']},
        {'id': 2, 'title': '닭가슴살 샐러드', 'ingredients': ['닭가슴살', '양상추', '토마토']},
        {'id': 3, 'title': '콩나물 국', 'ingredients': ['콩나물', '대파', '마늘']},
        {'id': 4, 'title': '된장찌개', 'ingredients': ['두부', '된장', '애호박']} # 매칭 재료 없음
    ]

    # 2. 알고리즘 호출
    print(f"📅 기준일: {today}\n")
    print("📢 추천 레시피 결과 (유통기한 임박순)")
    print("=" * 50)
    
    recommendations = get_expiry_recommendations(my_pantry_db, all_recipes_db)
    
    for rank, item in enumerate(recommendations, 1):
        print(f"{rank}위. [{item['total_score']}점] {item['title']}")
        print(f"   └─ 매칭된 급한 재료: {item['matched_reason']}")
        print("-" * 50)

📅 기준일: 2025-12-01

📢 추천 레시피 결과 (유통기한 임박순)
1위. [20.0점] 매운 어묵 콩나물찜
   └─ 매칭된 급한 재료: 어묵(10.0점), 콩나물(10.0점)
--------------------------------------------------
2위. [10.0점] 콩나물 국
   └─ 매칭된 급한 재료: 콩나물(10.0점)
--------------------------------------------------
3위. [2.5점] 닭가슴살 샐러드
   └─ 매칭된 급한 재료: 닭가슴살(2.5점)
--------------------------------------------------
